# GAN evaluation + image generation (Kaggle T4 x2)

Inputs: the train notebook's outputs (one per FRACTION/AUG) and the dataset `aptos-dr-256-genai`.
Per final pickle: 500 images per rare grade (seeds 0-499, psi 0.8) with `Synthetic=true` metadata,
overall FID/KID vs the val split (`calc_metrics.py`), per-class KID for grades 3/4
(`src/gan/metrics_per_class.py`). Outputs: `synth_{tag}.zip`, `gan_metrics.csv`, `samples_{tag}.png`.
Download on the lab PC with `kaggle kernels output <user>/<this-notebook> -p downloads/` then
`python -m src.gan.import_synth downloads/synth_f*.zip`.

In [ ]:
REPO = "https://github.com/nabirakhan/genai"
DATASET_DIR = "/kaggle/input/aptos-dr-256-genai"
N_PER_CLASS, PSI, CLASSES = 500, 0.8, "3,4"
ONLY_TAGS = None          # e.g. ["f100-ada"]; None = every tag found under /kaggle/input
GPUS = 2

In [ ]:
import os, sys, glob, re, json, time, shutil, signal, subprocess
WORK = "/kaggle/working"; CODE = f"{WORK}/genai"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "--recurse-submodules", REPO, CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "--recurse-submodules"], check=False)
os.chdir(CODE)
subprocess.run([sys.executable, "scripts/apply_patches.py"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "click", "requests", "tqdm", "ninja",
                "imageio-ffmpeg", "psutil", "scipy", "pyspng"], check=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch; print("torch", torch.__version__, "gpus", torch.cuda.device_count())
def sh(cmd, **kw):
    print("$", " ".join(cmd)); return subprocess.run(cmd, check=True, **kw)


In [ ]:
def total_kimg(path):
    m = re.search(r"-from(\d+)[/\\].*snapshot-(\d+)", path); return int(m.group(1)) + int(m.group(2)) if m else 0
best = {}
for s in glob.glob("/kaggle/input/**/network-snapshot-*.pkl", recursive=True):
    m = re.search(r"-(f\d{3}-(?:ada|noaug))-from", s)
    if m and (ONLY_TAGS is None or m.group(1) in ONLY_TAGS):
        if m.group(1) not in best or total_kimg(s) > total_kimg(best[m.group(1)]):
            best[m.group(1)] = s
os.makedirs(f"{WORK}/pkls", exist_ok=True)
PKLS = {}
for tag, s in sorted(best.items()):
    dst = f"{WORK}/pkls/{tag}.pkl"           # copy: calc_metrics must not write next to a read-only input
    if not os.path.exists(dst): shutil.copy(s, dst)
    PKLS[tag] = dst; print(f"{tag}: {total_kimg(s)} kimg  <- {s}")
assert PKLS, "no snapshots found; attach the train notebook outputs"
val = f"{DATASET_DIR}/aptos256_val"
REAL_VAL = val if os.path.isdir(val) else val + ".zip"
print("REAL_VAL =", REAL_VAL)

In [ ]:
import pandas as pd
overall = []
for tag, pkl in PKLS.items():
    out = f"{WORK}/synth_{tag}"
    sh([sys.executable, "-m", "src.gan.generate", "--pkl", pkl, "--outdir", out, "--classes", CLASSES, "--n", str(N_PER_CLASS), "--psi", str(PSI), "--tag", tag])
    r = subprocess.run([sys.executable, "third_party/stylegan3/calc_metrics.py", "--metrics=fid50k_full,kid50k_full", f"--network={pkl}", f"--data={REAL_VAL}", f"--gpus={GPUS}"], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-800:])
    for line in r.stdout.splitlines():
        if line.startswith("{") and '"results"' in line:
            res = json.loads(line)["results"]
            overall.append(dict(gan=tag, grade="all", n_real=None, n_gen=50000, fid=res.get("fid50k_full"), kid=res.get("kid50k_full"), kid_std=None))
    sh([sys.executable, "-m", "src.gan.metrics_per_class", "--gan", tag, "--synth-dir", out, "--real", REAL_VAL, "--classes", CLASSES, "--out", f"{WORK}/gan_metrics.csv"])
    shutil.make_archive(out, "zip", out)
df = pd.read_csv(f"{WORK}/gan_metrics.csv")
df = pd.concat([df[df.grade.astype(str) != "all"], pd.DataFrame(overall)], ignore_index=True)
df.to_csv(f"{WORK}/gan_metrics.csv", index=False); print(df.to_string(index=False))

In [ ]:
import numpy as np
from PIL import Image
sys.path.insert(0, CODE)
from src.gan.watermark import add_visible_label
for tag in PKLS:
    rows = []
    for g in CLASSES.split(","):
        files = sorted(glob.glob(f"{WORK}/synth_{tag}/g{g}/*.png"))[:8]
        rows.append(np.concatenate([np.asarray(add_visible_label(Image.open(f).convert("RGB"))) for f in files], axis=1))
    grid = Image.fromarray(np.concatenate(rows, axis=0)); grid.save(f"{WORK}/samples_{tag}.png"); display(grid.resize((grid.width // 2, grid.height // 2)))
for tag in PKLS: shutil.rmtree(f"{WORK}/synth_{tag}", ignore_errors=True)   # zips remain
os.chdir(WORK)
shutil.rmtree(CODE, ignore_errors=True)
print(sorted(os.listdir(WORK)))